# ROBERT UI Notebook Launcher

Use this notebook to install UI dependencies, launch the local Dash app, and stop it cleanly.

This notebook is intended for users who are comfortable running Jupyter notebooks and want a guided way to run the app.

## Step 1: Configure Paths and Environment

This cell resolves the project root and confirms that required files exist.

Expected output:
- The project root path
- Confirmation that `start_ui.py` and `requirements_ui.txt` were found

In [87]:
# Resolve imports and config values before launching the app.
from pathlib import Path
import os
import sys

# Resolve project root by walking up from this notebook location.
possible_roots = [
    Path.cwd(),
    Path.cwd().parent,
    Path.cwd().parent.parent,
    Path.cwd().parent.parent.parent,
]

PROJECT_ROOT = None
for candidate in possible_roots:
    if (candidate / 'AGENTS.md').exists() and (candidate / 'start_ui.py').exists():
        PROJECT_ROOT = candidate.resolve()
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError('Could not locate project root containing AGENTS.md and start_ui.py')

START_UI = PROJECT_ROOT / 'start_ui.py'
REQS_UI = PROJECT_ROOT / 'requirements_ui.txt'

# Ensure UI config module is importable from notebook context.
sys.path.insert(0, str(PROJECT_ROOT / 'agent' / 'ui'))
from config import load_api_key, load_openai_model, load_response_style, load_chat_mode

# Unified key availability check (env OR config/.env via UI loader).
api_key_loaded = load_api_key()
api_key_source = 'none'
if os.getenv('ROBERT_CHAT_API_KEY', '').strip():
    api_key_source = 'environment variable (ROBERT_CHAT_API_KEY)'
elif api_key_loaded:
    api_key_source = 'agent/ui/config/.env'

# OpenAI model check with source resolution.
openai_model = load_openai_model('gpt-4o-mini')
model_source = 'default (gpt-4o-mini)'
if os.getenv('ROBERT_OPENAI_MODEL', '').strip():
    model_source = 'environment variable (ROBERT_OPENAI_MODEL)'
else:
    local_env = PROJECT_ROOT / 'agent' / 'ui' / 'config' / '.env'
    if local_env.exists():
        for line in local_env.read_text(encoding='utf-8', errors='ignore').splitlines():
            line = line.strip()
            if line.startswith('ROBERT_OPENAI_MODEL=') and line.split('=', 1)[1].strip():
                model_source = 'agent/ui/config/.env'
                break

# Response style and chat mode checks.
response_style = load_response_style('REPORT_ONLY')
response_style_source = 'default (REPORT_ONLY)'
if os.getenv('ROBERT_RESPONSE_STYLE', '').strip():
    response_style_source = 'environment variable (ROBERT_RESPONSE_STYLE)'
else:
    local_env = PROJECT_ROOT / 'agent' / 'ui' / 'config' / '.env'
    if local_env.exists():
        for line in local_env.read_text(encoding='utf-8', errors='ignore').splitlines():
            if line.strip().startswith('ROBERT_RESPONSE_STYLE='):
                response_style_source = 'agent/ui/config/.env'
                break

chat_mode = load_chat_mode('HEURISTICS_FIRST')
chat_mode_source = 'default (HEURISTICS_FIRST)'
if os.getenv('ROBERT_CHAT_MODE', '').strip():
    chat_mode_source = 'environment variable (ROBERT_CHAT_MODE)'
else:
    local_env = PROJECT_ROOT / 'agent' / 'ui' / 'config' / '.env'
    if local_env.exists():
        for line in local_env.read_text(encoding='utf-8', errors='ignore').splitlines():
            if line.strip().startswith('ROBERT_CHAT_MODE='):
                chat_mode_source = 'agent/ui/config/.env'
                break

print(f'Project root: {PROJECT_ROOT}')
print(f'start_ui.py found: {START_UI.exists()} -> {START_UI}')
print(f'requirements_ui.txt found: {REQS_UI.exists()} -> {REQS_UI}')
print(f'API key available to UI: {bool(api_key_loaded)}')
print(f'API key source: {api_key_source}')
print(f'OpenAI model for chat fallback: {openai_model}')
print(f'Model source: {model_source}')
print(f'Response style: {response_style}')
print(f'Response style source: {response_style_source}')
print(f'Chat mode: {chat_mode}')
print(f'Chat mode source: {chat_mode_source}')

Project root: /Users/cjcscha/ROBERT/helper_rob/robert
start_ui.py found: True -> /Users/cjcscha/ROBERT/helper_rob/robert/start_ui.py
requirements_ui.txt found: True -> /Users/cjcscha/ROBERT/helper_rob/robert/requirements_ui.txt
API key available to UI: True
API key source: agent/ui/config/.env
OpenAI model for chat fallback: gpt-4o-mini
Model source: agent/ui/config/.env
Response style: REPORT_ONLY
Response style source: default (REPORT_ONLY)
Chat mode: HEURISTICS_FIRST
Chat mode source: default (HEURISTICS_FIRST)


## Step 2: Install/Update UI Dependencies

Run this cell to install dependencies from `requirements_ui.txt` into the active notebook Python environment.

You can skip this if already installed, but running it is safe.

In [88]:
import importlib
import subprocess
import sys

cmd = [sys.executable, '-m', 'pip', 'install', '-r', str(REQS_UI)]
print('Running:', ' '.join(cmd))
result = subprocess.run(cmd, cwd=str(PROJECT_ROOT), check=False, capture_output=True, text=True)
print('Exit code:', result.returncode)

if result.returncode == 0:
    print('Dependency installation completed successfully.')
else:
    # Only continue if core UI dependencies can still be imported in this environment.
    required_modules = [
        ('dash', 'dash'),
        ('dash_bootstrap_components', 'dash-bootstrap-components'),
        ('plotly', 'plotly'),
        ('openai', 'openai'),
    ]
    missing = []
    for module_name, package_name in required_modules:
        try:
            importlib.import_module(module_name)
        except Exception as exc:
            missing.append(f"{package_name} import failed: {exc}")

    if missing:
        print('pip stderr (tail):')
        stderr_tail = '\n'.join((result.stderr or '').splitlines()[-20:])
        print(stderr_tail if stderr_tail else '[no stderr captured]')
        missing_block = '\n - '.join(missing)
        raise RuntimeError(
            'Dependency installation failed and required packages are missing.\n'
            f' - {missing_block}'
        )

    print('WARNING: pip install reported an error, but core UI dependencies are importable.')
    print('Proceeding with existing environment as dependencies appear already present.')
    stderr_tail = '\n'.join((result.stderr or '').splitlines()[-10:])
    if stderr_tail:
        print('pip stderr (tail):')
        print(stderr_tail)

Running: /Users/cjcscha/mambaforge/envs/robert/bin/python -m pip install -r /Users/cjcscha/ROBERT/helper_rob/robert/requirements_ui.txt
Exit code: 1
Proceeding with existing environment as dependencies appear already present.
pip stderr (tail):
ERROR: Could not install packages due to an OSError: [Errno 2] No such file or directory: '/Users/cjcscha/mambaforge/envs/robert/lib/python3.10/site-packages/certifi-2026.1.4.dist-info/METADATA'


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


## Step 3: (Optional) Configure API Key and Model for Chat

If you want to enable the OpenAI chat fallback, set ROBERT_CHAT_API_KEY before running the notebook.

Do NOT paste your key into this notebook. Instead, set it one of these ways:
1. Environment variable: `export ROBERT_CHAT_API_KEY=sk-...` (in your shell before launching the notebook)
2. Local file: create `agent/ui/config/.env` with `ROBERT_CHAT_API_KEY=sk-...` (git-ignored)

**Optional Configuration Options:**

**Model Override:**
- Environment variable: `export ROBERT_OPENAI_MODEL=gpt-4o-mini`
- Or add to `agent/ui/config/.env`: `ROBERT_OPENAI_MODEL=gpt-4o-mini`

**Response Instruction Style:** (Toggleable in UI)
- `REPORT_ONLY` (default): Concise, evidence-focused explanations
- `REPORT_WITH_KB`: Includes reference material from knowledge base
- Environment variable: `export ROBERT_RESPONSE_STYLE=REPORT_ONLY` or `REPORT_WITH_KB`
- Or add to `agent/ui/config/.env`: `ROBERT_RESPONSE_STYLE=REPORT_WITH_KB`

**Chat Routing Mode:** (Toggleable in UI)
- `HEURISTICS_FIRST` (default): Try deterministic FAQ rules first, then LLM fallback
- `LLM_ONLY`: Always use LLM API (requires ROBERT_CHAT_API_KEY and OpenAI connection)
- Environment variable: `export ROBERT_CHAT_MODE=HEURISTICS_FIRST` or `LLM_ONLY`
- Or add to `agent/ui/config/.env`: `ROBERT_CHAT_MODE=LLM_ONLY`

If not set, the app still runs with full diagnostic display, but chat will show a fallback message when no deterministic/local answer is available.


In [89]:
# Check whether API key and model are available to the UI loader (env OR config/.env).
# DO NOT EDIT THIS CELL TO PASTE YOUR KEY.

api_key_loaded = load_api_key()
api_key_source = 'none'
if os.getenv('ROBERT_CHAT_API_KEY', '').strip():
    api_key_source = 'environment variable (ROBERT_CHAT_API_KEY)'
elif api_key_loaded:
    api_key_source = 'agent/ui/config/.env'

openai_model = load_openai_model('gpt-4o-mini')
model_source = 'default (gpt-4o-mini)'
if os.getenv('ROBERT_OPENAI_MODEL', '').strip():
    model_source = 'environment variable (ROBERT_OPENAI_MODEL)'
else:
    local_env = PROJECT_ROOT / 'agent' / 'ui' / 'config' / '.env'
    if local_env.exists():
        for line in local_env.read_text(encoding='utf-8', errors='ignore').splitlines():
            line = line.strip()
            if line.startswith('ROBERT_OPENAI_MODEL=') and line.split('=', 1)[1].strip():
                model_source = 'agent/ui/config/.env'
                break

if api_key_loaded:
    print('✓ API key is available. Chat fallback will be enabled.')
    print(f'  Source: {api_key_source}')
else:
    print('⚠ API key is not available. Chat will show a fallback message.')
    print('  Set ROBERT_CHAT_API_KEY in your shell or create agent/ui/config/.env before running the UI.')

print(f'OpenAI model for chat fallback: {openai_model}')
print(f'Model source: {model_source}')

✓ API key is available. Chat fallback will be enabled.
  Source: agent/ui/config/.env
OpenAI model for chat fallback: gpt-4o-mini
Model source: agent/ui/config/.env


## Step 4: Stop Any Existing UI Server

This cell stops a previously running UI server from this notebook session before starting a fresh one.

If no server is running, it will print a message and continue safely.


In [93]:
if 'ui_proc' in globals() and ui_proc is not None and ui_proc.poll() is None:
    ui_proc.terminate()
    try:
        ui_proc.wait(timeout=5)
        print(f'UI process {ui_proc.pid} terminated with code {ui_proc.returncode}')
    except Exception:
        ui_proc.kill()
        print(f'UI process {ui_proc.pid} force-killed')
else:
    print('No running UI process found in this notebook session. Continuing to start.')
ui_proc = None


UI process 69492 terminated with code -15


## Step 5: Start the UI Server

This starts `start_ui.py` in the background and prints startup logs.

Expected output includes a URL like `http://127.0.0.1:8050`.


In [94]:
import subprocess
import sys
import time

ui_proc = subprocess.Popen(
    [sys.executable, str(START_UI)],
    cwd=str(PROJECT_ROOT),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

print(f'Started UI process with PID: {ui_proc.pid}')
print('Streaming startup logs for up to 15 seconds...')

deadline = time.time() + 15
while time.time() < deadline:
    line = ui_proc.stdout.readline()
    if line:
        print(line.rstrip())
        if 'Running on http://127.0.0.1:8050' in line or 'Dash is running on http://127.0.0.1:8050/' in line:
            print('UI appears ready.')
            break
    if ui_proc.poll() is not None:
        raise RuntimeError(f'UI process exited early with code {ui_proc.returncode}')
else:
    print('Startup window ended. If no errors were shown, try opening the URL below.')

UI_URL = 'http://127.0.0.1:8050'
print('Open in browser:', UI_URL)


Started UI process with PID: 33107
Streaming startup logs for up to 15 seconds...
[2026-05-26 05:40:09,713] INFO: API key loaded from agent/ui/config/.env
[2026-05-26 05:40:09,713] INFO: Project root: /Users/cjcscha/ROBERT/helper_rob/robert
[2026-05-26 05:40:09,713] INFO: Run archive: /Users/cjcscha/ROBERT/helper_rob/robert/agent/run_archive
[2026-05-26 05:40:09,713] INFO: API key: sk-proj-Rl...
[2026-05-26 05:40:09,732] INFO: Found 8 diagnostic runs
[2026-05-26 05:40:09,749] INFO: Starting app on 127.0.0.1:8050
[2026-05-26 05:40:09,749] INFO: Open http://127.0.0.1:8050 in your browser
[2026-05-26 05:40:09,749] INFO: Press Ctrl+C to stop
Dash is running on http://127.0.0.1:8050/
UI appears ready.
Open in browser: http://127.0.0.1:8050


## Step 6: Open the UI in Your Browser

This attempts to open the local app automatically.


In [95]:
import webbrowser

url = globals().get('UI_URL', 'http://127.0.0.1:8050')
opened = webbrowser.open(url)
print('Attempted to open:', url)
print('Browser open call returned:', opened)

Attempted to open: http://127.0.0.1:8050
Browser open call returned: True


## Troubleshooting

- **Port in use (8050)**: Stop any previous server process, then rerun Step 4.
- **No runs found**: Ensure archived runs with `run_context.json` and `diagnosis_summary.md` exist in `agent/run_archive/`.
- **Chat unavailable**: This is expected without `ROBERT_CHAT_API_KEY` set.